In [69]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.distributions.normal import Normal

import gymnasium as gym
import numpy as np

import random
import os

save_dir = "./model"
os.makedirs(save_dir, exist_ok=True)

ENV_NAME = ["HalfCheetah-v5"]

# ENV_NAME = ["HalfCheetah-v5",
#             "Ant-v5",
#             "Hopper-v5",
#             "Humanoid-v5",
#             "HumanoidStandup-v5",
#             "InvertedDoublePendulum-v5",
#             "InvertedPendulum-v5",
#             "Pusher-v5",
#             "Reacher-v5",
#             "Swimmer-v5",
#             "Walker2d-v5"]

In [70]:

class ContinuousGaussianPolicy(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.mean_head = nn.Linear(64, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim))  # 학습 가능한 std

    def forward(self, obs):
        x = self.net(obs)
        action_mean = self.mean_head(x)
        action_std = torch.exp(self.log_std)
        return Normal(action_mean, action_std)  # Normal 분포 반환
        
    def act(self, obs):
        with torch.no_grad():
            dist = self.forward(obs)
            action = dist.sample()
            return action.clamp(-1.0, 1.0), dist.log_prob(action)

In [71]:
class ValueNetwork(nn.Module):
    def __init__(self, obs_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, obs):
        return self.net(obs).squeeze(-1)

In [72]:
class RolloutBuffer:
    def __init__(self, max_size, state_dim, action_dim, device):
        self.device = device
    
        self.max_size = max_size
        self.ptr = 0  # 현재 인덱스
        self.size = 0  # 실제 저장된 개수

        self.states = np.zeros((max_size, state_dim), dtype=np.float32)
        self.actions = np.zeros((max_size, action_dim), dtype=np.float32)
        self.rewards = np.zeros((max_size, 1), dtype=np.float32)
        self.next_states = np.zeros((max_size, state_dim), dtype=np.float32)
        self.dones = np.zeros((max_size, 1), dtype=np.float32)

    def add(self, transition):
        s, a, r, s_next, d = transition

        self.states[self.ptr] = s
        self.actions[self.ptr] = a
        self.rewards[self.ptr] = r
        self.next_states[self.ptr] = s_next
        self.dones[self.ptr] = d

        self.ptr += 1
        self.size = min(self.size + 1, self.max_size)

    def sample(self):
        idx = slice(0, self.size) 
        return (
            torch.from_numpy(self.states[idx]).float().to(self.device),
            torch.from_numpy(self.actions[idx]),
            torch.from_numpy(self.rewards[idx]),
            torch.from_numpy(self.next_states[idx]),
            torch.from_numpy(self.dones[idx])
        )

    def reset(self):
        self.ptr = 0
        self.size = 0

In [73]:

class TRPO:
    def __init__(self, obs_dim, act_dim):
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.policy = ContinuousGaussianPolicy(obs_dim, act_dim)
        self.old_policy = ContinuousGaussianPolicy(obs_dim, act_dim)
        self.value_network = ValueNetwork(obs_dim)
        self.value_optim = torch.optim.Adam(self.value_network.parameters(), lr = 3e-3)
        
        self.trajectories = RolloutBuffer(max_size=1000, state_dim=obs_dim, action_dim=act_dim, device=self.device)
        
        self.backtrack_coeff = 1.0
        self.backtrack_alpha = 0.5
        self.t = 0
    
    def kl_divergence(self, obs, old_policy, new_policy):
        old_mu, old_std = old_policy(obs)
        old_mu, old_std = old_mu.detach(), old_std.detach()
        
        new_mu, new_std = new_policy(obs)
        KL = torch.log(new_std / old_std) + (old_std.pow(2) + (old_mu - new_mu).pow(2)) / (2.0 * new_std.pow(2)) - 0.5
        
        return KL.sum(-1, keepdim=True).mean()
    
    def flat_grad(self, grads, hessian=False):
        grad_flatten = []
        if not hessian:
            for grad in grads:
                grad_flatten.append(grad.view(-1))
            grad_flatten = torch.cat(grad_flatten)

        else:
            for grad in grads:
                grad_flatten.append(grad.contiguous().view(-1))
            grad_flatten = torch.cat(grad_flatten).data
        return grad_flatten
    
    def fim_product(self, obs, v):
        v.detach()
        
        kl = self.kl_divergence(obs, self.old_policy, self.policy)
        kl_grad = torch.autograd.grad(kl, self.policy.parameters())
        kl_grad = self.flat_grad(kl_grad) # 왜 flattening 하지
        
        kl_grad_v = (kl_grad * v).sum()
        kl_hessian = torch.autograd.grad(kl_grad_v, self.policy.parameters())
        kl_hessian = self.flat_grad(kl_hessian, hessian=True)
        return kl_hessian + v
    
    def cg_gradient(self, obs, b, iter = 10, threshold = 1e-10):
        x = torch.zeros_like(b)
        r = b.clone()
        v = r.clone()
        
        for _ in range(iter):
            Av = self.fim_product(obs, v)
            
            t = torch.dot(r, r) / (torch.dot(v, Av))
            x = x + t * v
            
            r_new = r - t * Av
            
            s = torch.dot(r_new, r_new) / torch.dot(r, r)
            v = r_new + s * v
            r = r_new
            
            if torch.dot(r, r) < threshold:
                break
        return x
    
    def update_model(self, model, new_params):
        index = 0
        for params in model.parameters():
            params_length = len(params.view(-1))
            new_param = new_params[index: index + params_length]
            new_param = new_param.view(params.size())
            params.data.copy_(new_param)
            index += params_length
    
    def train(self):
        states, actions, rewards, next_states, dones = self.trajectories.sample()
        
        # GAE 및 returns 계산
        with torch.no_grad():
            values = self.value_network(states).squeeze(-1)            # (B,)
            next_values = self.value_network(next_states).squeeze(-1)  # (B,)
            deltas = rewards + self.gamma * (1 - dones) * next_values - values

            advantages = torch.zeros_like(rewards)
            gae = 0.0
            for t in reversed(range(len(rewards))):
                gae = deltas[t] + self.gamma * self.lamda * (1 - dones[t]) * gae
                advantages[t] = gae

            returns = advantages + values
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        for _ in range(100):
            # Value function update
            values_pred = self.value_network(states).squeeze(-1)
            value_loss = F.mse_loss(values_pred, returns)
            self.value_optimizer.zero_grad()
            value_loss.backward()
            self.value_optimizer.step()

        # update policy network
        with torch.no_grad():
            dist = self.policy(states)
            old_log_probs = dist.log_prob(actions).sum(-1)
            
        dist = self.policy(states)
        log_probs = dist.log_prob(actions).sum(-1)
        
        # Policy Loss
        ratio_old = (log_probs - old_log_probs).exp()
        policy_loss_old = (ratio_old * advantages).mean()
        
        # The Gradient of Surrogate Loss 'g'
        gradient = torch.autograd.grad(policy_loss_old, self.policy.parameters())
        gradient = self.flat_grad(gradient)
        
        # s = F-1g using cg
        s = self.congugate_gradient(states, gradient.data)
        gHg = (self.fisher_vector_product(states, s) * s).sum(0)
        gHg = -gHg if gHg < 0 else gHg
        step_size = torch.sqrt(2 * 0.01 / gHg)
        old_params = self.flat_params(self.policy)
        self.update_model(self.old_policy, old_params)
        
        # Backtrack line search
        with torch.no_grad():
            expected_improve = (gradient * step_size * s).sum(0, keepdim=True)
            for i in range(10):
                params = old_params + self.backtrack_coeff * step_size * s
                self.update_model(self.policy, params)

                mu, std = self.policy(s)
                m = Normal(mu, std)
                z = torch.atanh(torch.clamp(a, -1.0 + 1e-7, 1.0 - 1e-7))
                log_prob = m.log_prob(z).sum(dim=-1, keepdim=True)
                ratio = (log_prob - old_log_probs).exp()
                policy_loss = (ratio * advantages).mean()

                loss_improve = policy_loss - policy_loss_old
                expected_improve *= self.backtrack_coeff
                improve_condition = loss_improve / expected_improve

                kl = self.gaussian_kl(s, self.old_policy, self.policy)

                if kl < 0.01 and improve_condition > self.backtrack_alpha:
                    break

                if i == 9:
                    params = self.flat_params(self.old_policy)
                    self.update_model(self.policy, params)
                self.backtrack_coeff *= 0.5

        result = {
            'Step': self.t,
            'policy_loss': policy_loss.item(),
            'value_loss': value_loss.item(),
        }

        return result
            
         
    def step(self, transition):
        result = None
        self.t += 1
        self.trajectories.add(transition)
        if self.trajectories.size >= 1000:
            result = self.train()

        return result
             
        

In [74]:
def evaluate(env_name, agent, seed, eval_iterations):
    env = gym.make(env_name)
    scores = []
    for i in range(eval_iterations):
        (s, _), terminated, truncated, score = env.reset(seed=seed + 100 + i), False, False, 0
        while not (terminated or truncated):
            a = agent.act(s, training=False)
            s_prime, r, terminated, truncated, _ = env.step(2.0 * a)
            score += r
            s = s_prime
        scores.append(score)
    env.close()
    return round(np.mean(scores), 4)

In [75]:
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

In [ ]:
from tqdm import tqdm

env_name = 'Pendulum-v1'

seed = 0
seed_all(seed)
hidden_dims = (64, 64, )
max_iterations = 1000000
eval_intervals = 10000
eval_iterations = 10
batch_size = 2000
activation_fn = torch.tanh
gamma = 0.95
lmda = 0.95
backtrack_alpha = 0.5

env = gym.make(env_name)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
agent = TRPO(
    state_dim,
    action_dim
)

logger = []
(s, _), terminated, truncated = env.reset(seed=seed), False, False
for t in tqdm(range(1, max_iterations + 1)):
    a = agent.policy.act(torch.from_numpy(s).float())
    a_np=np.array(a)
    s_prime, r, terminated, truncated, _ = env.step(a_np)
    result = agent.step((s, a, r, s_prime, terminated))
    s = s_prime
    
    if result is not None:
        logger.append([t, 'policy_loss', result['policy_loss']])
        logger.append([t, 'value_loss', result['value_loss']])
    
    if terminated or truncated:
        (s, _), terminated, truncated = env.reset(), False, False
        
    if t % eval_intervals == 0:
        score = evaluate(env_name, agent, seed, eval_iterations)
        logger.append([t, 'Avg return', score])

  0%|          | 0/1000000 [00:00<?, ?it/s]


AttributeError: 'tuple' object has no attribute 'numpy'